# Unified RL-Assisted MPC Penalty Multipliers

This notebook unifies the TD3/SAC and nominal/disturbance MPC penalty-multiplier experiments behind two top-level switches: `AGENT_KIND` and `RUN_MODE`.

In [ ]:
from pathlib import Path
import os

from systems.polymer.data_io import canonical_baseline_path
from utils.notebook_setup import prepare_polymer_notebook_env, print_notebook_summary

# User-editable agent and run controls.
AGENT_KIND = "td3"   # "td3" | "sac"
RUN_MODE = "nominal"  # "nominal" | "disturb"
STATE_MODE = "standard"  # "standard" | "mismatch"
STYLE_PROFILE = "hybrid"  # "hybrid" | "paper" | "debug"
SAVE_PDF = False

# Optional directory overrides. Leave as None to use Polymer/Data and Polymer/Results.
POLYMER_DATA_DIR_OVERRIDE = None
POLYMER_RESULTS_DIR_OVERRIDE = None

REPO_ROOT, DATA_DIR, RESULT_DIR = prepare_polymer_notebook_env(
    data_dir_override=POLYMER_DATA_DIR_OVERRIDE,
    results_dir_override=POLYMER_RESULTS_DIR_OVERRIDE,
)
os.chdir(REPO_ROOT)

RUN_PROFILE_MAP = {
    ("td3", "nominal"): {"baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "nominal", data_override=POLYMER_DATA_DIR_OVERRIDE), "result_prefix": "td3_weights_nominal", "compare_prefix": "nominal_compare_td3_weights", "compare_mode": "nominal", "plot_start_episode": 2, "compare_start_episode": 2},
    ("td3", "disturb"): {"baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "disturb", data_override=POLYMER_DATA_DIR_OVERRIDE), "result_prefix": "td3_weights_disturb", "compare_prefix": "disturb_compare_td3_weights", "compare_mode": "disturb", "plot_start_episode": 2, "compare_start_episode": 2},
    ("sac", "nominal"): {"baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "nominal", data_override=POLYMER_DATA_DIR_OVERRIDE), "result_prefix": "sac_weights_nominal", "compare_prefix": "nominal_compare_sac_weights", "compare_mode": "nominal", "plot_start_episode": 2, "compare_start_episode": 2},
    ("sac", "disturb"): {"baseline_mpc_path": canonical_baseline_path(REPO_ROOT, "disturb", data_override=POLYMER_DATA_DIR_OVERRIDE), "result_prefix": "sac_weights_disturb", "compare_prefix": "disturb_compare_sac_weights", "compare_mode": "disturb", "plot_start_episode": 2, "compare_start_episode": 2},
}
RUN_PROFILE = RUN_PROFILE_MAP[(AGENT_KIND, RUN_MODE)]
RESULT_PREFIX = RUN_PROFILE["result_prefix"]
COMPARE_PREFIX = RUN_PROFILE["compare_prefix"]
BASELINE_MPC_PATH = RUN_PROFILE["baseline_mpc_path"]

print_notebook_summary(
    "Polymer weight-multiplier configuration",
    {
        "Polymer data dir": DATA_DIR,
        "Polymer results": RESULT_DIR,
        "Agent kind": AGENT_KIND,
        "Run mode": RUN_MODE,
        "State mode": STATE_MODE,
        "Baseline MPC": BASELINE_MPC_PATH,
    },
)


In [ ]:
import numpy as np
import torch

from SACAgent.sac_agent import SACAgent
from Simulation.mpc import MpcSolverGeneral
from Simulation.system_functions import PolymerCSTR
from TD3Agent.agent import TD3Agent
from systems.polymer import (
    POLYMER_DELTA_T_HOURS,
    POLYMER_DESIGN_PARAMS,
    POLYMER_INPUT_BOUNDS,
    POLYMER_OBSERVER_POLES,
    POLYMER_RL_SETPOINTS_PHYS,
    POLYMER_SETPOINT_RANGE_PHYS,
    POLYMER_SYSTEM_METADATA,
    POLYMER_SS_INPUTS,
    POLYMER_SYSTEM_PARAMS,
    RL_REWARD_DEFAULTS,
    load_polymer_system_data,
)
from utils.helpers import apply_min_max
from utils.observer import compute_observer_gain
from utils.plotting import compare_mpc_rl_from_dirs, plot_weight_multiplier_results
from utils.rewards import make_reward_fn_relative_QR
from utils.state_features import get_rl_state_dim
from utils.weights_runner import run_weight_multiplier_supervisor


In [ ]:
# Build the polymer plant and load the canonical polymer data bundle.
system_params = POLYMER_SYSTEM_PARAMS.copy()
system_design_params = POLYMER_DESIGN_PARAMS.copy()
system_steady_state_inputs = POLYMER_SS_INPUTS.copy()
delta_t = POLYMER_DELTA_T_HOURS

cstr_ss = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
steady_states = {"ss_inputs": cstr_ss.ss_inputs, "y_ss": cstr_ss.y_ss}

setpoint_y = POLYMER_SETPOINT_RANGE_PHYS.copy()
u_min = POLYMER_INPUT_BOUNDS["u_min"].copy()
u_max = POLYMER_INPUT_BOUNDS["u_max"].copy()
system_data = load_polymer_system_data(
    REPO_ROOT,
    steady_states=steady_states,
    setpoint_y=setpoint_y,
    u_min=u_min,
    u_max=u_max,
    n_inputs=2,
    data_override=POLYMER_DATA_DIR_OVERRIDE,
)

A_aug = system_data["A_aug"]
B_aug = system_data["B_aug"]
C_aug = system_data["C_aug"]
data_min = system_data["data_min"]
data_max = system_data["data_max"]
min_max_dict = system_data["min_max_dict"]

inputs_number = 2
y_sp_scenario = POLYMER_RL_SETPOINTS_PHYS.copy()
y_sp_scenario = (
    apply_min_max(y_sp_scenario, data_min[inputs_number:], data_max[inputs_number:])
    - apply_min_max(steady_states["y_ss"], data_min[inputs_number:], data_max[inputs_number:])
)


In [ ]:
N_INPUTS = int(B_aug.shape[1])
N_OUTPUTS = int(C_aug.shape[0])
STATE_DIM = get_rl_state_dim(A_aug.shape[0], N_OUTPUTS, N_INPUTS, STATE_MODE)
ACTION_DIM = 4
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

predict_h = 9
cont_h = 3
Q1_penalty = 5.0
Q2_penalty = 1.0
R1_penalty = 1.0
R2_penalty = 1.0
LOW_COEF = np.array([0.5, 0.5, 0.5, 0.5])
HIGH_COEF = np.array([3.0, 3.0, 3.0, 3.0])
ACTOR_FREEZE = 0

MPC_obj = MpcSolverGeneral(
    A_aug,
    B_aug,
    C_aug,
    Q_out=np.array([Q1_penalty, Q2_penalty], float),
    R_in=np.array([R1_penalty, R2_penalty], float),
    NP=predict_h,
    NC=cont_h,
)

k_rel = np.array([0.003, 0.0003])
band_floor_phys = np.array([0.006, 0.07])
Q_diag = np.array([518.0, 90.0])
R_diag = np.array([90.0, 90.0])
reward_params, reward_fn = make_reward_fn_relative_QR(
    data_min,
    data_max,
    N_INPUTS,
    k_rel,
    band_floor_phys,
    Q_diag,
    R_diag,
    tau_frac=0.7,
    gamma_out=0.5,
    gamma_in=0.5,
    beta=7.0,
    gate="geom",
    lam_in=1.0,
    bonus_kind="exp",
    bonus_k=12.0,
    bonus_p=0.6,
    bonus_c=20.0,
)
print(reward_params)

nominal_qs = 459.0
nominal_qi = 108.0
nominal_hA = 1.05e6
qi_change = 0.85
qs_change = 1.3
ha_change = 0.85

if AGENT_KIND == "td3":
    weight_agent = TD3Agent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=[512, 512, 512, 512, 512],
        critic_hidden=[512, 512, 512, 512, 512],
        gamma=0.995,
        actor_lr=1e-4,
        critic_lr=1e-4,
        batch_size=128,
        policy_delay=2,
        target_policy_smoothing_noise_std=0.1,
        noise_clip=0.2,
        max_action=1.0,
        tau=0.005,
        std_start=0.2,
        std_end=0.02,
        std_decay_rate=0.99995,
        std_decay_mode="exp",
        buffer_size=50_000,
        device=DEVICE,
        actor_freeze=ACTOR_FREEZE,
    )
elif AGENT_KIND == "sac":
    weight_agent = SACAgent(
        state_dim=STATE_DIM,
        action_dim=ACTION_DIM,
        actor_hidden=[512, 512, 512, 512, 512],
        critic_hidden=[512, 512, 512, 512, 512],
        gamma=0.995,
        actor_lr=1e-4,
        critic_lr=1e-4,
        alpha_lr=1e-4,
        batch_size=128,
        grad_clip_norm=10.0,
        init_alpha=0.01,
        learn_alpha=True,
        target_entropy=-ACTION_DIM,
        target_update="soft",
        tau=0.005,
        hard_update_interval=10_000,
        activation="relu",
        use_layernorm=False,
        dropout=0.0,
        max_action=1.0,
        buffer_size=50_000,
        use_per=True,
        device=DEVICE,
        use_adamw=True,
        actor_freeze=ACTOR_FREEZE,
    )
else:
    raise ValueError("AGENT_KIND must be 'td3' or 'sac'.")

print(f"State dimension: {STATE_DIM}")
print(weight_agent.__class__.__name__)

In [ ]:
weight_cfg = {
    "agent_kind": AGENT_KIND,
    "run_mode": RUN_MODE,
    "state_mode": STATE_MODE,
    "n_tests": n_tests,
    "set_points_len": set_points_len,
    "warm_start": warm_start,
    "test_cycle": TEST_CYCLE,
    "predict_h": predict_h,
    "cont_h": cont_h,
    "low_coef": LOW_COEF,
    "high_coef": HIGH_COEF,
    "nominal_qi": nominal_qi,
    "nominal_qs": nominal_qs,
    "nominal_ha": nominal_hA,
    "qi_change": qi_change,
    "qs_change": qs_change,
    "ha_change": ha_change,
    "Q1_penalty": Q1_penalty,
    "Q2_penalty": Q2_penalty,
    "R1_penalty": R1_penalty,
    "R2_penalty": R2_penalty,
    "b_min": system_data["b_min"],
    "b_max": system_data["b_max"],
}

cstr = PolymerCSTR(system_params, system_design_params, system_steady_state_inputs, delta_t)
runtime_ctx = {
    "system": cstr,
    "agent": weight_agent,
    "MPC_obj": MPC_obj,
    "steady_states": steady_states,
    "min_max_dict": min_max_dict,
    "data_min": data_min,
    "data_max": data_max,
    "A_aug": A_aug,
    "B_aug": B_aug,
    "C_aug": C_aug,
    "poles": poles,
    "y_sp_scenario": y_sp_scenario,
    "reward_fn": reward_fn,
    "system_metadata": POLYMER_SYSTEM_METADATA,
    "reward_params": reward_params,
    "system_metadata": POLYMER_SYSTEM_METADATA,
}

result_bundle = run_weight_multiplier_supervisor(weight_cfg=weight_cfg, runtime_ctx=runtime_ctx)
result_bundle["mpc_path_or_dir"] = RUN_PROFILE["baseline_mpc_path"]

In [ ]:
out_dir_rl = plot_weight_multiplier_results(
    result_bundle=result_bundle,
    plot_cfg={
        "directory": RESULT_DIR,
        "prefix_name": RESULT_PREFIX,
        "start_episode": RUN_PROFILE["plot_start_episode"],
        "save_pdf": SAVE_PDF,
    },
)

out_dir_cmp = compare_mpc_rl_from_dirs(
    rl_dir=out_dir_rl,
    mpc_path_or_dir=RUN_PROFILE["baseline_mpc_path"],
    reward_fn=reward_fn,
    directory=RESULT_DIR,
    prefix_name=COMPARE_PREFIX,
    compare_mode=RUN_PROFILE["compare_mode"],
    start_episode=COMPARE_START_EPISODE,
)

print("RL result directory:", out_dir_rl)
print("Comparison directory:", out_dir_cmp)